# Modelo de predição de Revenue

Este notebook carrega o dataset de seeds/online_shoppers_intention.csv, treina um modelo simples e salva os artefatos para uso posterior.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

DATASET_PATH = Path("seeds/online_shoppers_intention.csv")
MODEL_PATH = Path("models/random_forest_model.joblib")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

shoppers_df = pd.read_csv(DATASET_PATH)
shoppers_df.head()

In [ ]:
# Conversão de tipos para o treino
shoppers_df["Weekend"] = shoppers_df["Weekend"].astype(bool)
shoppers_df["Revenue"] = shoppers_df["Revenue"].astype(bool)

# Colunas categóricas e numéricas
categorical_features = ["Month", "VisitorType"]
numeric_features = [c for c in shoppers_df.columns if c not in categorical_features + ["Revenue"]]

X = shoppers_df.drop(columns=["Revenue"])
y = shoppers_df["Revenue"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42, n_estimators=200, max_depth=8)),
    ]
)

model.fit(X_train, y_train)

In [ ]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", round(accuracy, 4))
print(classification_report(y_test, predictions))

In [ ]:
joblib.dump(model, MODEL_PATH)
print("Modelo salvo em", MODEL_PATH)